# TP : comptage du nombre d'occurrences de mots

On compare trois méthodes : Python simple, pseudo-MapReduce en Python et Hadoop MapReduce, sur trois fichiers de tailles différentes.

## 1. Création des fichiers de test

On crée 3 fichiers : un petit (1 000 lignes), un moyen (100 000 lignes) et un grand (1 000 000 lignes). Chaque ligne contient 10 mots pris au hasard dans une liste.

In [1]:
import random
import time

mots = ["chat", "chien", "arbre", "fleur", "maison", "soleil", "montagne", "livre",
        "ciel", "mer", "lune", "route", "oiseau", "jardin", "voiture"]

fichiers = {"mots_petit.txt": 1000, "mots_moyen.txt": 100000, "mots_grand.txt": 1000000}

for nom, n in fichiers.items():
    with open(nom, "w", encoding="utf-8") as f:
        for i in range(n):
            f.write(" ".join(random.choices(mots, k=10)) + "\n")

!ls -lh mots_*.txt

-rw-r--r-- 1 exert exert  60M Oct 13 11:38 mots_grand.txt
-rw-r--r-- 1 exert exert 6.0M Oct 13 11:38 mots_moyen.txt
-rw-r--r-- 1 exert exert  62K Oct 13 11:38 mots_petit.txt


## 2. Version 1 : Python simple

In [2]:
def compter_occurrences(fichier):
    with open(fichier, "r", encoding="utf-8") as f:
        mots = f.read().split()

    compteur = {}
    for mot in mots:
        compteur[mot] = compteur.get(mot, 0) + 1
    return compteur


temps_v1 = {}
for nom in fichiers:
    debut = time.time()
    resultats = compter_occurrences(nom)
    temps_v1[nom] = time.time() - debut
    print(nom, ":", round(temps_v1[nom], 3), "s")

mots_petit.txt : 0.005 s
mots_moyen.txt : 0.218 s
mots_grand.txt : 2.46 s


In [3]:
# Résultat sur le petit fichier
for mot, nb in sorted(compter_occurrences("mots_petit.txt").items(), key=lambda x: x[1], reverse=True):
    print(f"{mot} : {nb}")

ciel : 722
arbre : 693
route : 686
oiseau : 677
maison : 676
livre : 673
chien : 673
chat : 671
fleur : 663
soleil : 659
voiture : 657
lune : 650
mer : 645
montagne : 641
jardin : 614


Le fichier est lu d'un coup puis on parcourt les mots une seule fois en mettant à jour un dictionnaire. Le temps est donc à peu près proportionnel au nombre de mots : quand le fichier est 10 fois plus gros, le temps est environ 10 fois plus long. Tout se passe en mémoire sur une seule machine, sans étape intermédiaire, c'est pour ça que c'est très rapide.

## 3. Version 2 : pseudo-MapReduce en Python

In [4]:
from collections import defaultdict


def mapper(ligne):
    mots = ligne.strip().split()
    return [(mot, 1) for mot in mots]


def shuffle(mapped_data):
    grouped = defaultdict(list)
    for mot, valeur in mapped_data:
        grouped[mot].append(valeur)
    return grouped


def reducer(grouped_data):
    reduced = {}
    for mot, valeurs in grouped_data.items():
        reduced[mot] = sum(valeurs)
    return reduced


def pseudo_mapreduce(fichier):
    mapped = []
    with open(fichier, "r", encoding="utf-8") as f:
        for ligne in f:
            mapped.extend(mapper(ligne))
    grouped = shuffle(mapped)
    return reducer(grouped)


temps_v2 = {}
for nom in fichiers:
    debut = time.time()
    resultats = pseudo_mapreduce(nom)
    temps_v2[nom] = time.time() - debut
    print(nom, ":", round(temps_v2[nom], 3), "s")

print("Même résultat que la version 1 :", pseudo_mapreduce("mots_petit.txt") == compter_occurrences("mots_petit.txt"))

mots_petit.txt : 0.006 s
mots_moyen.txt : 0.384 s
mots_grand.txt : 4.45 s
Même résultat que la version 1 : True


On retrouve les étapes de MapReduce : le mapper transforme chaque ligne en paires (mot, 1), le shuffle regroupe les paires par mot et le reducer fait la somme. Le résultat est le même que la version 1 mais c'est plus lent, parce qu'on crée une paire pour chaque mot puis une liste de 1 pour chaque mot avant d'additionner. Ces étapes servent à répartir le travail sur plusieurs machines, mais ici tout tourne sur une seule machine, donc ça ajoute juste du travail et de la mémoire.

## 4. Version 3 : Hadoop MapReduce

### Installation de Java et Hadoop

On télécharge Java 11 et Hadoop 3.4.3 dans le dossier personnel.

In [5]:
import os

HOME = os.path.expanduser("~")
os.environ["JAVA_HOME"] = HOME + "/jdk11"
os.environ["HADOOP_HOME"] = HOME + "/hadoop"
os.environ["PATH"] = HOME + "/jdk11/bin:" + HOME + "/hadoop/bin:" + os.environ["PATH"]

In [6]:
!mkdir -p ~/jdk11 ~/hadoop
!wget -qO- "https://api.adoptium.net/v3/binary/latest/11/ga/linux/x64/jdk/hotspot/normal/eclipse" | tar -xz -C ~/jdk11 --strip-components=1
!wget -qO- https://dlcdn.apache.org/hadoop/common/hadoop-3.4.3/hadoop-3.4.3.tar.gz | tar -xz -C ~/hadoop --strip-components=1
!java -version
!hadoop version

openjdk version "11.0.32.1" 2026-08-18
OpenJDK Runtime Environment Temurin-11.0.32.1+1 (build 11.0.32.1+1)
OpenJDK 64-Bit Server VM Temurin-11.0.32.1+1 (build 11.0.32.1+1, mixed mode)
Hadoop 3.4.3
Source code repository https://github.com/apache/hadoop.git -r 9d50c6884666e794e45102260a4017bb31802e1b
Compiled by stevel on 2026-02-13T14:23Z
Compiled on platform linux-x86_64
Compiled with protoc 3.23.4
From source with checksum 2331238c4c2929e66645316a32a8613
This command was run using /home/exert/hadoop/share/hadoop/common/hadoop-common-3.4.3.jar


### Configuration en mode pseudo-distribué

Tous les services de Hadoop tournent sur la même machine, avec une réplication de 1 puisqu'il n'y a qu'un seul DataNode.

In [53]:
%%writefile ~/hadoop/etc/hadoop/core-site.xml
<configuration>
  <property><name>fs.defaultFS</name><value>hdfs://localhost:8020</value></property>
  <property><name>hadoop.tmp.dir</name><value>${user.home}/hadoop_tmp</value></property>
</configuration>

Overwriting /home/exert/hadoop/etc/hadoop/core-site.xml


In [54]:
%%writefile ~/hadoop/etc/hadoop/hdfs-site.xml
<configuration>
  <property><name>dfs.replication</name><value>1</value></property>
</configuration>

Overwriting /home/exert/hadoop/etc/hadoop/hdfs-site.xml


In [55]:
%%writefile ~/hadoop/etc/hadoop/mapred-site.xml
<configuration>
  <property><name>mapreduce.framework.name</name><value>yarn</value></property>
  <property><name>mapreduce.application.classpath</name><value>$HADOOP_MAPRED_HOME/share/hadoop/mapreduce/*:$HADOOP_MAPRED_HOME/share/hadoop/mapreduce/lib/*</value></property>
</configuration>

Overwriting /home/exert/hadoop/etc/hadoop/mapred-site.xml


In [56]:
%%writefile ~/hadoop/etc/hadoop/yarn-site.xml
<configuration>
  <property><name>yarn.nodemanager.aux-services</name><value>mapreduce_shuffle</value></property>
  <property><name>yarn.nodemanager.env-whitelist</name><value>JAVA_HOME,HADOOP_COMMON_HOME,HADOOP_HDFS_HOME,HADOOP_CONF_DIR,CLASSPATH_PREPEND_DISTCACHE,HADOOP_YARN_HOME,HADOOP_HOME,PATH,LANG,TZ,HADOOP_MAPRED_HOME</value></property>
  <property><name>yarn.nodemanager.vmem-check-enabled</name><value>false</value></property>
</configuration>

Overwriting /home/exert/hadoop/etc/hadoop/yarn-site.xml


### Démarrage de HDFS et YARN

On formate HDFS (seulement la première fois) puis on lance les 4 services : NameNode et DataNode pour HDFS, ResourceManager et NodeManager pour YARN.

In [57]:
!yarn --daemon stop nodemanager
!yarn --daemon stop resourcemanager
!hdfs --daemon stop datanode
!hdfs --daemon stop namenode
!rm -f /tmp/hadoop-*.pid
!rm -rf ~/hadoop_tmp ~/hadoop/logs/*

!hdfs namenode -format -nonInteractive 2>&1 | tail -1
!setsid -w hdfs --daemon start namenode
!setsid -w hdfs --daemon start datanode
!setsid -w yarn --daemon start resourcemanager
!setsid -w yarn --daemon start nodemanager
!hdfs dfsadmin -safemode wait
!jps

************************************************************/
Safe mode is OFF
12352 Jps
11905 ResourceManager
11777 DataNode
11701 NameNode
11989 NodeManager


In [58]:
!grep -hE "ERROR|Exception" ~/hadoop/logs/*namenode*.log | tail -5
!grep -hE "ERROR|Exception" ~/hadoop/logs/*resourcemanager*.log | tail -5

### Exécution du WordCount

On copie les 3 fichiers dans HDFS puis on lance le job `wordcount` fourni avec Hadoop sur chaque fichier.

In [59]:
!hdfs dfs -mkdir -p /wordcount/input
!hdfs dfs -put -f mots_*.txt /wordcount/input/
!hdfs dfs -ls /wordcount/input

Found 3 items
-rw-r--r--   1 exert supergroup   62662009 2026-10-13 12:00 /wordcount/input/mots_grand.txt
-rw-r--r--   1 exert supergroup    6266700 2026-10-13 12:00 /wordcount/input/mots_moyen.txt
-rw-r--r--   1 exert supergroup      62530 2026-10-13 12:00 /wordcount/input/mots_petit.txt


In [60]:
JAR = HOME + "/hadoop/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.4.3.jar"

temps_v3 = {}
for nom in fichiers:
    !hdfs dfs -rm -r -f /wordcount/output_{nom}
    debut = time.time()
    !hadoop jar {JAR} wordcount /wordcount/input/{nom} /wordcount/output_{nom} 2>&1 | grep -E "completed|Exception"
    temps_v3[nom] = time.time() - debut
    print(nom, ":", round(temps_v3[nom], 3), "s")

2026-10-13 12:01:43,986 INFO mapreduce.Job: Job job_1791885628342_0001 completed successfully
mots_petit.txt : 37.45 s
2026-10-13 12:02:23,607 INFO mapreduce.Job: Job job_1791885628342_0002 completed successfully
mots_moyen.txt : 36.263 s
2026-10-13 12:03:07,382 INFO mapreduce.Job: Job job_1791885628342_0003 completed successfully
mots_grand.txt : 40.608 s


In [61]:
# Résultat sur le petit fichier
!hdfs dfs -cat /wordcount/output_mots_petit.txt/part-r-00000

arbre	693
chat	671
chien	673
ciel	722
fleur	663
jardin	614
livre	673
lune	650
maison	676
mer	645
montagne	641
oiseau	677
route	686
soleil	659
voiture	657


On obtient les mêmes nombres qu'avec les versions Python. Par contre Hadoop met plusieurs secondes même pour le petit fichier : à chaque job il faut démarrer une JVM, envoyer le job à YARN, lancer les conteneurs pour les mappers et le reducer, puis lire et écrire dans HDFS. Ce temps de démarrage est presque le même quelle que soit la taille du fichier, c'est pour ça que le temps augmente peu entre le petit et le moyen fichier.

## 5. Comparaison

In [62]:
print("Fichier".ljust(16), "Python simple".rjust(14), "Pseudo-MR".rjust(12), "Hadoop".rjust(10))
for nom in fichiers:
    print(nom.ljust(16), f"{temps_v1[nom]:12.3f} s", f"{temps_v2[nom]:10.3f} s", f"{temps_v3[nom]:8.3f} s")

Fichier           Python simple    Pseudo-MR     Hadoop
mots_petit.txt          0.005 s      0.006 s   37.450 s
mots_moyen.txt          0.218 s      0.384 s   36.263 s
mots_grand.txt          2.460 s      4.450 s   40.608 s


Sur ces fichiers, les versions Python sont beaucoup plus rapides que Hadoop. Hadoop a un coût fixe de démarrage de plusieurs secondes et, comme tout tourne sur une seule machine, il ne peut pas profiter de la répartition du calcul. Il devient intéressant quand les données sont trop grosses pour tenir en mémoire sur une machine (plusieurs gigaoctets) : les versions Python chargent tout en mémoire, alors que Hadoop découpe le fichier en blocs et peut les traiter en parallèle sur plusieurs machines.

## 6. Arrêt de Hadoop

In [63]:
!yarn --daemon stop nodemanager
!yarn --daemon stop resourcemanager
!hdfs --daemon stop datanode
!hdfs --daemon stop namenode